# Clase 7 — Visión por computadora como herramienta

La mesa de ayuda puede recibir una fotografía del equipo o una captura. Hoy conectaremos un modelo preentrenado de imágenes como herramienta.

No entrenaremos una CNN. Aprenderemos qué recibe, qué devuelve y por qué sus etiquetas necesitan contexto.

## Objetivos

- Representar una imagen como píxeles y tensor.
- Comprender filtro, CNN y modelo preentrenado desde la práctica.
- Ejecutar MobileNet y leer top-k.
- Agregar umbral y control de dominio.
- Exponer visión con el mismo contrato de herramienta.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from sklearn.datasets import load_sample_image

imagenes={
 "flor":Image.fromarray(load_sample_image("flower.jpg")),
 "paisaje":Image.fromarray(load_sample_image("china.jpg")),
}
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,(nombre,img) in zip(axes,imagenes.items()):
    ax.imshow(img); ax.set_title(f"{nombre}: {img.size}"); ax.axis("off")
plt.tight_layout()

---
## 1. Una imagen es una matriz

Una imagen RGB contiene tres valores por píxel: rojo, verde y azul. Alto, ancho y canales forman un tensor.

La computadora no recibe la palabra flor. Recibe números.

In [ ]:
imagen=imagenes["flor"]
array=np.array(imagen)
print("Forma:",array.shape)
print("Tipo:",array.dtype)
print("Píxel [100,100]:",array[100,100])
print("Mínimo y máximo:",array.min(),array.max())

---
## 2. De filtros a CNN

Un filtro recorre zonas pequeñas y responde a patrones como bordes o texturas. Una CNN aprende muchos filtros y combina sus señales en capas.

    píxeles → bordes → texturas → partes → representación → probabilidades

Las primeras capas detectan patrones simples; las posteriores combinan información más abstracta.

In [ ]:
gris=np.array(imagen.convert("L").resize((300,200)),dtype=float)
# Diferencia horizontal sencilla: resalta cambios entre píxeles vecinos.
bordes=np.abs(np.diff(gris,axis=1))
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].imshow(gris,cmap="gray"); axes[0].set_title("Escala de grises")
axes[1].imshow(bordes,cmap="magma"); axes[1].set_title("Cambios horizontales")
for ax in axes: ax.axis("off")
plt.tight_layout()

---
## 3. MobileNet preentrenada

MobileNet aprendió a clasificar categorías de ImageNet. Sus pesos pueden descargarse una vez y ejecutarse en CPU.

El modo real queda desactivado por defecto para que el notebook sea independiente de internet. Las salidas guardadas están marcadas como tales.

In [ ]:
USAR_MODELO_VISION=False
modelo_vision=None
pesos=None

if USAR_MODELO_VISION:
    import torch
    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
    pesos=MobileNet_V3_Small_Weights.DEFAULT
    modelo_vision=mobilenet_v3_small(weights=pesos).eval()
    print("MobileNet lista")
else:
    print("Modo aula: se analizarán salidas precargadas de MobileNet.")

In [ ]:
SALIDAS_VISION_AULA={
 "flor":[("daisy",0.61),("rapeseed",0.16),("bee",0.05)],
 "paisaje":[("palace",0.31),("monastery",0.18),("valley",0.08)],
}

def clasificar_imagen(nombre,imagen,top_k=3):
    if modelo_vision is None:
        predicciones = SALIDAS_VISION_AULA.get(nombre)
        if predicciones is None:
            return {"modo":"sin_salida_precargada",
                    "predicciones":[("clase desconocida", 0.0)]}
        return {"modo":"precargado",
                "predicciones":predicciones[:top_k]}
    import torch
    entrada=pesos.transforms()(imagen).unsqueeze(0)
    with torch.no_grad():
        probs=modelo_vision(entrada).softmax(dim=1)[0]
    valores,indices=probs.topk(top_k)
    pred=[(pesos.meta["categories"][i],round(float(v),3))
          for v,i in zip(valores,indices)]
    return {"modo":"modelo_real","predicciones":pred}

for nombre,img in imagenes.items():
    print(nombre,"→",clasificar_imagen(nombre,img))

### Cómo leer top-k

El modelo devuelve las categorías más probables dentro de su vocabulario. 0.61 no significa 61% de certeza universal: es una probabilidad relativa entre las clases conocidas por el modelo.

Si la imagen pertenece a otro dominio, el modelo igualmente elegirá alguna etiqueta.

---
## 4. El problema de dominio

ImageNet reconoce objetos cotidianos, pero no fue entrenado para diagnosticar errores de nuestra aplicación. Puede ayudar a describir un objeto; no puede concluir la causa técnica de una falla.

Por eso separamos:

- observación visual;
- descripción de la persona;
- decisión de soporte.

In [ ]:
def herramienta_vision(nombre,imagen,umbral=0.50):
    salida=clasificar_imagen(nombre,imagen)
    etiqueta,confianza=salida["predicciones"][0]
    confiable=confianza>=umbral
    return {
      "ok":True,
      "datos":{"etiqueta":etiqueta,"confianza":confianza,
               "top_k":salida["predicciones"],
               "suficiente":confiable},
      "modo":salida["modo"],
    }

herramienta_vision("flor",imagenes["flor"])

---
## 5. Integrar visión al agente

Ambos agentes usan la misma herramienta. La diferencia sigue estando en cómo deciden. Ninguno debe transformar una etiqueta visual en diagnóstico automático.

In [ ]:
def agente_multimodal(descripcion,nombre_imagen,imagen,umbral=0.50):
    vision=herramienta_vision(nombre_imagen,imagen,umbral)
    datos=vision["datos"]
    if not datos["suficiente"]:
        decision="revision_humana"
        respuesta="La imagen no permite una observación confiable."
    else:
        decision="continuar_diagnostico"
        respuesta=(f"El modelo observa posiblemente '{datos['etiqueta']}'. "
                   f"Necesito relacionarlo con: {descripcion}")
    return {"categoria":"incidente","decision":decision,
            "herramienta":"vision","respuesta":respuesta,
            "confianza":datos["confianza"],
            "requiere_revision":decision=="revision_humana",
            "traza":[{"paso":"vision","resultado":vision}]}

agente_multimodal("Adjunto una imagen del problema",
                  "flor",imagenes["flor"])

---
## 6. Umbral y costo del error

Un umbral alto deriva más casos. Uno bajo permite más automatización, pero acepta predicciones débiles. La elección depende de la consecuencia, no solo de la exactitud.

In [ ]:
for umbral in [0.20,0.50,0.80]:
    salida=agente_multimodal("Imagen adjunta","paisaje",
                             imagenes["paisaje"],umbral)
    print(umbral,"→",salida["decision"],
          salida["confianza"])

---
## 📝 Actividad 1 — Inspeccionar una imagen propia

Cargá una foto de un periférico o equipo, mostrala, imprimí forma y rango de píxeles. Si activás MobileNet, registrá top-3 y modo de ejecución.

In [ ]:
# RUTA_PROPIA = "mi_imagen.jpg"
# propia = Image.open(RUTA_PROPIA).convert("RGB")
# print(np.array(propia).shape)
# plt.imshow(propia); plt.axis("off")

observaciones={"objeto_esperado":"...","top_1":"...",
               "confianza":"...","coincide":"..."}
observaciones

---
## 📝 Actividad 2 — Diseñar el umbral

Probá ambas imágenes con tres umbrales. Elegí uno para una sugerencia informativa y otro para una acción con impacto. Justificá la diferencia.

In [ ]:
tabla=[]
for nombre,img in imagenes.items():
    for umbral in [0.20,0.50,0.80]:
        r=herramienta_vision(nombre,img,umbral)["datos"]
        tabla.append({"imagen":nombre,"umbral":umbral,
                      "etiqueta":r["etiqueta"],
                      "confianza":r["confianza"],
                      "aceptada":r["suficiente"]})
import pandas as pd
pd.DataFrame(tabla)

---
## 📝 Actividad 3 — Caso fuera de dominio

La persona adjunta una captura de un error, pero MobileNet devuelve monitor o sitio web. Diseñá la respuesta del agente sin inventar el mensaje de error. Debe pedir un dato útil y decidir si deriva.

In [ ]:
caso_fuera_dominio={
 "descripcion_usuario":"No funciona",
 "etiqueta_modelo":"monitor",
 "confianza":0.72,
 "dato_adicional_a_pedir":"TODO",
 "decision":"TODO",
 "respuesta_segura":"TODO",
}
caso_fuera_dominio

---
## ✅ Resumen

Vimos píxeles, filtros, CNN, pesos preentrenados, top-k, umbral y cambio de dominio. La visión quedó encapsulada como herramienta y nunca como diagnóstico automático.

En la Clase 8 modelaremos decisiones repetidas mediante estados, acciones y recompensas.